<!-- NB_DOC_INTRO_V1 -->
# DuckDB — SQL analytique embarque

> 📚 **Doc thematique** : [docs/07_BDD_DE.md](docs/07_BDD_DE.md) (Bases de donnees / Data Engineering / Web)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

**DuckDB** = 'SQLite pour l'analytique' : moteur SQL columnar **in-process** (pas de serveur), ultra-rapide sur Parquet / CSV / DataFrame. Standard 2024-2025 pour single-machine analytical workloads (jusqu'a TB).

Ce notebook execute DuckDB sur des DataFrames Pandas et fichiers Parquet, compare vs pandas pur, et montre les features killer (window functions, ASOF joins, requete directe sur Parquet/CSV).

## Plan

1. Installation + concepts
2. Connection + requete simple
3. Requete sur DataFrame pandas (sans copie)
4. Comparatif perf vs pandas pur
5. Lecture directe Parquet / CSV
6. Window functions
7. ASOF join (TS)
8. Persistence
9. Pieges + References


In [ ]:
import duckdb
import pandas as pd
import numpy as np
import time, os, tempfile
import warnings
warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
print(f"DuckDB : {duckdb.__version__}")

## 1. Connection + requete simple

In [ ]:
# In-memory (jetable)
con = duckdb.connect(":memory:")

# Requete SQL
result = con.sql("SELECT 1 + 1 AS sum, 'hello' AS msg, current_date AS today")
print(result)

# .fetchall() / .df() pour materialiser
df_out = con.sql("SELECT range AS i, range*2 AS double FROM range(5)").df()
print(f"\nAs DataFrame :\n{df_out}")

## 2. Requete sur DataFrame pandas (zero copie !)

**Killer feature** : DuckDB peut requeter un DataFrame **directement** par son nom de variable Python.

In [ ]:
# Dataset jouet
df = pd.DataFrame({
    "user_id": np.random.randint(1, 100, 10_000),
    "category": np.random.choice(["A", "B", "C", "D"], 10_000),
    "amount": np.random.exponential(100, 10_000),
    "ts": pd.date_range("2024-01-01", periods=10_000, freq="1h"),
})
print(f"DataFrame : {df.shape}")

# Requete SQL directe sur df (le nom 'df' est resolu automatiquement)
result = duckdb.sql('''
    SELECT category,
           COUNT(*) AS n,
           SUM(amount) AS total,
           AVG(amount) AS mean,
           QUANTILE_CONT(amount, 0.95) AS p95
    FROM df
    GROUP BY category
    ORDER BY total DESC
''').df()
print(result.round(2))

## 3. Comparatif perf vs pandas pur

In [ ]:
n = 1_000_000
big = pd.DataFrame({
    "grp": np.random.choice(["A", "B", "C", "D"], n),
    "val": np.random.randn(n),
    "qty": np.random.randint(1, 100, n),
})

# Pandas
t0 = time.perf_counter()
res_pd = big.groupby("grp").agg(mean=("val", "mean"), sum_qty=("qty", "sum"), n=("val", "count"))
t_pd = time.perf_counter() - t0

# DuckDB
t0 = time.perf_counter()
res_duck = duckdb.sql('''
    SELECT grp, AVG(val) AS mean, SUM(qty) AS sum_qty, COUNT(*) AS n
    FROM big
    GROUP BY grp
''').df()
t_duck = time.perf_counter() - t0

print(f"Pandas  : {t_pd*1000:7.1f} ms")
print(f"DuckDB  : {t_duck*1000:7.1f} ms  ({t_pd/t_duck:.1f}x vs pandas)")

## 4. Lecture directe Parquet / CSV

In [ ]:
# Ecrire un fichier Parquet
tmp_parquet = os.path.join(tempfile.gettempdir(), "demo.parquet")
big.to_parquet(tmp_parquet)
print(f"Parquet ecrit : {os.path.getsize(tmp_parquet) / 1024**2:.1f} MB")

# Requeter DIRECTEMENT le fichier (zero loading en Python)
result = duckdb.sql(f'''
    SELECT grp, COUNT(*) AS n, AVG(val) AS mean_val
    FROM read_parquet('{tmp_parquet}')
    WHERE val > 0
    GROUP BY grp
''').df()
print(f"\nQuery sur Parquet direct :\n{result.round(3)}")

# Wildcard sur plusieurs fichiers
# duckdb.sql("SELECT * FROM read_parquet('data/*.parquet')")

# CSV idem
# duckdb.sql("SELECT * FROM read_csv_auto('data.csv')")

os.remove(tmp_parquet)

## 5. Window functions — analytics SQL

In [ ]:
result = duckdb.sql('''
    SELECT user_id, ts, amount,
           SUM(amount) OVER (PARTITION BY user_id ORDER BY ts) AS cum_amount_user,
           AVG(amount) OVER (PARTITION BY category) AS avg_in_cat,
           RANK() OVER (PARTITION BY category ORDER BY amount DESC) AS rank_in_cat,
           LAG(amount, 1) OVER (PARTITION BY user_id ORDER BY ts) AS prev_amount,
           amount - LAG(amount, 1) OVER (PARTITION BY user_id ORDER BY ts) AS delta
    FROM df
    LIMIT 10
''').df()
print(result.round(2))

## 6. ASOF join (TS — joindre 'la plus proche par le bas')

In [ ]:
# Prix par minute, trades par jour
prices = pd.DataFrame({
    "ts": pd.date_range("2024-01-01 09:00", "2024-01-01 09:10", freq="1min"),
    "price": [100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110],
})

trades = pd.DataFrame({
    "ts": pd.to_datetime(["2024-01-01 09:02:30", "2024-01-01 09:05:45", "2024-01-01 09:09:15"]),
    "qty": [10, 5, 20],
})

# Pour chaque trade, recuperer le dernier prix connu
result = duckdb.sql('''
    SELECT t.ts AS trade_ts, t.qty, p.ts AS price_ts, p.price
    FROM trades t
    ASOF LEFT JOIN prices p ON t.ts >= p.ts
''').df()
print(result)

## 7. Persistence — base sur disque

In [ ]:
db_path = os.path.join(tempfile.gettempdir(), "mydb.duckdb")
con_disk = duckdb.connect(db_path)

# Creer une table persistante
con_disk.sql("CREATE OR REPLACE TABLE users AS SELECT * FROM df")

# Verifier
n = con_disk.sql("SELECT COUNT(*) AS n FROM users").fetchone()[0]
print(f"Persistent users : {n}")
con_disk.close()

# Re-ouvrir
con2 = duckdb.connect(db_path)
print(f"After reopen : {con2.sql('SELECT COUNT(*) FROM users').fetchone()[0]}")
con2.close()
os.remove(db_path)
print("Cleanup OK")

## 8. Pieges et anti-patterns

| ❌ Anti-pattern | ✅ Correctif |
|---|---|
| Loader Parquet en pandas avant requete | DuckDB peut requeter `read_parquet('file')` direct |
| `to_pandas()` sur 100M lignes pour requete simple | Tout faire en SQL |
| Pas exploiter window functions | DuckDB SQL est tres riche |
| Concurrent writes single file | DuckDB ne supporte pas le multi-writer (single process) |
| Croire DuckDB = remplace OLTP | C'est OLAP : pas adapte pour 1000 writes/sec |
| `LIMIT 5` sans `ORDER BY` | Non deterministe |
| Comparer perf sans `WHERE` | DuckDB shine avec filters / aggregations |

## 9. Quand utiliser DuckDB vs alternatives

| Cas | Recommandation |
|---|---|
| Analytics single-machine, < 1 TB | **DuckDB** (le meilleur 2024-2025) |
| < 100k lignes, simple | pandas |
| 100k - 10M, expressions complexes | polars |
| > 1 TB ou multi-node | Spark / BigQuery |
| OLTP (1000s writes/sec) | Postgres |
| Embedded mobile / edge | SQLite |


## References

### Documentation
- DuckDB docs : https://duckdb.org/docs/
- DuckDB Python : https://duckdb.org/docs/api/python/overview

### Voir aussi
- [Structures_DataFrame.ipynb](Structures_DataFrame.ipynb)
- [Structures_Polars.ipynb](Structures_Polars.ipynb)
- [Structure_BDD_&_DataFrame.ipynb](Structure_BDD_&_DataFrame.ipynb)
- [DE_PySpark.ipynb](DE_PySpark.ipynb)
